# Baseline Flux Balance Analysis (FBA)

In this notebook, we evaluate our engineered *E. coli* strain under baseline wild-type conditions using **Flux Balance Analysis (FBA)**. 

### What is FBA calculating?
FBA solves a linear programming problem to maximize an objective function - by default, cellular growth (biomass production). We will measure two critical variables:
1. **Growth Rate ($h^{-1}$):** How fast the bacteria reproduces.
2. **Naringenin Flux ($\text{mmol/gDW/h}$):** How fast our target molecule is harvested through the demand reaction (`DM_naring_c`).

In [1]:
import cobra

#Loading our saved model (with a fallback if working directory varies)
try:
    model = cobra.io.load_json_model("../data/iJO1366_naringenin.json")
except FileNotFoundError:
    model = cobra.io.load_json_model("iJO1366_naringenin.json")

#Ensuring the objective function is set to maximize biomass (cell growth)
model.objective = 'BIOMASS_Ec_iJO1366_WT_53p95M'

#Running the Flux Balance Analysis simulation
solution = model.optimize()

#Printing the simulation outcome
print(f"Simulation Status: {solution.status}")
print(f"Wild-Type Growth Rate: {solution.objective_value:.4f} /h")

#Querying the flux through our heterologous Naringenin pathway drain
naringenin_flux = solution.fluxes['DM_naring_c']
print(f"Baseline Naringenin Flux: {naringenin_flux:.4f} mmol/gDW/h")

Simulation Status: optimal
Wild-Type Growth Rate: 0.9865 /h
Baseline Naringenin Flux: 0.0000 mmol/gDW/h


### Theoretical Maximum Naringenin Yield

Calculating the absolute maximum amount of Naringenin this cell structure could produce if all resources were diverted to the plant pathway, by setting our objective function directly to `DM_naring_c`.

In [2]:
#Setting objective to maximize Naringenin production
model.objective = 'DM_naring_c'

#Running FBA
max_naringenin_sol = model.optimize()

#1.Putting objective value as Naringenin Flux (mmol/gDW/h)
max_naring_flux = max_naringenin_sol.objective_value

#2.checking growth rate, by explicitly querying the biomass reaction
growth_at_max_yield = max_naringenin_sol.fluxes['BIOMASS_Ec_iJO1366_WT_53p95M']

print(f"Theoretical Max Naringenin Flux: {max_naring_flux:.4f} mmol/gDW/h")
print(f"Growth Rate during Max Production: {growth_at_max_yield:.4f} /h")

Theoretical Max Naringenin Flux: 3.3599 mmol/gDW/h
Growth Rate during Max Production: 0.0000 /h
